In [7]:
# Load Phase 7 data and persisted modeling contracts

import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_poisson_deviance,
)


DATA_PATH = Path(
    "../data/engineered/"
    "ronaldo_1000_ml_engineered.csv"
)

ARTIFACT_DIR = Path("../artifacts")


# Load engineered historical dataset
df = pd.read_csv(
    DATA_PATH,
    parse_dates=["date"],
)

df = (
    df.sort_values("date")
    .reset_index(drop=True)
)


# Load persisted contracts
with open(
    ARTIFACT_DIR / "feature_spec.json",
    "r",
    encoding="utf-8",
) as f:
    feature_spec = json.load(f)


with open(
    ARTIFACT_DIR / "temporal_folds.json",
    "r",
    encoding="utf-8",
) as f:
    temporal_spec = json.load(f)


with open(
    ARTIFACT_DIR / "validation_policy.json",
    "r",
    encoding="utf-8",
) as f:
    validation_policy = json.load(f)


with open(
    ARTIFACT_DIR / "preprocessing_spec.json",
    "r",
    encoding="utf-8",
) as f:
    preprocessing_spec = json.load(f)


with open(
    ARTIFACT_DIR / "baseline_model_spec.json",
    "r",
    encoding="utf-8",
) as f:
    baseline_spec = json.load(f)


# Restore modeling contracts
FULL_FEATURES = (
    feature_spec["full_features"]
)

NUMERIC_FEATURES = (
    feature_spec["numeric_features"]
)

CATEGORICAL_FEATURES = (
    feature_spec["categorical_features"]
)

TARGET = (
    feature_spec["primary_target"]
)

SECONDARY_TARGETS = (
    feature_spec["secondary_targets"]
)

# temporal_folds.json was saved as a list of fold dictionaries
temporal_folds_spec = temporal_spec

# Final test boundary comes from the validation policy artifact
FINAL_TEST_START = pd.Timestamp(
    validation_policy["final_test_start"]
)

FEATURE_SETS = (
    preprocessing_spec["feature_sets"]
)


# Locked benchmark from Phase 6
POISSON_BASELINE_MACRO = float(
    baseline_spec[
        "selected_statistical_baseline"
    ]["macro_metrics"]["poisson_deviance"]
)

POISSON_BASELINE_POOLED = float(
    baseline_spec[
        "selected_statistical_baseline"
    ]["pooled_metrics"]["poisson_deviance"]
)


# Development/test boundary
development_df = df[
    df["date"] < FINAL_TEST_START
].copy()

final_test_df = df[
    df["date"] >= FINAL_TEST_START
].copy()


print("PHASE 7 — MAIN MODELS")
print("=" * 75)

print(
    "Dataset:",
    df.shape
)

print(
    "Career:",
    df["date"].min().date(),
    "→",
    df["date"].max().date()
)

print(
    "\nDevelopment pool:",
    len(development_df)
)

print(
    "Development period:",
    development_df["date"].min().date(),
    "→",
    development_df["date"].max().date()
)

print(
    "\nFinal test matches:",
    len(final_test_df)
)

print(
    "Final test period:",
    final_test_df["date"].min().date(),
    "→",
    final_test_df["date"].max().date()
)

print(
    "\nFull features:",
    len(FULL_FEATURES)
)

print(
    "Numeric features:",
    len(NUMERIC_FEATURES)
)

print(
    "Categorical features:",
    len(CATEGORICAL_FEATURES)
)

print(
    "Feature sets:",
    {
        name: len(features)
        for name, features
        in FEATURE_SETS.items()
    }
)

print(
    "\nPrimary target:",
    TARGET
)

print(
    "Temporal folds:",
    len(temporal_folds_spec)
)

print(
    "\nLOCKED BASELINE TO BEAT"
)

print(
    "Macro Poisson deviance :",
    round(
        POISSON_BASELINE_MACRO,
        6
    )
)

print(
    "Pooled Poisson deviance:",
    round(
        POISSON_BASELINE_POOLED,
        6
    )
)


# Contract checks
assert df["match_id"].nunique() == 1333
assert len(df) == 1333
assert df[TARGET].sum() == 978
assert df["date"].is_monotonic_increasing

assert len(FULL_FEATURES) == 84
assert len(NUMERIC_FEATURES) == 77
assert len(CATEGORICAL_FEATURES) == 7

assert len(development_df) == 1255
assert len(final_test_df) == 78

assert len(temporal_folds_spec) == 4

assert abs(
    POISSON_BASELINE_MACRO
    - 1.113978
) < 1e-5


print(
    "\nPhase 7 contracts: PASSED"
)

PHASE 7 — MAIN MODELS
Dataset: (1333, 100)
Career: 2002-08-14 → 2026-08-28

Development pool: 1255
Development period: 2002-08-14 → 2024-12-06

Final test matches: 78
Final test period: 2025-01-09 → 2026-08-28

Full features: 84
Numeric features: 77
Categorical features: 7
Feature sets: {'full': 84, 'context': 80, 'no_season': 83}

Primary target: target_goals
Temporal folds: 4

LOCKED BASELINE TO BEAT
Macro Poisson deviance : 1.113978
Pooled Poisson deviance: 1.104881

Phase 7 contracts: PASSED


## First CatBoost Poisson Model

In [10]:
# First CatBoost Poisson model on temporal folds

from catboost import CatBoostRegressor


def evaluate_catboost_poisson(
    feature_set_name="no_season",
    params=None,
):
    features = FEATURE_SETS[feature_set_name]

    categorical_features = [
        feature
        for feature in features
        if feature in CATEGORICAL_FEATURES
    ]

    fold_results = []
    oof_parts = []

    default_params = {
        "loss_function": "Poisson",
        "iterations": 500,
        "depth": 5,
        "learning_rate": 0.03,
        "l2_leaf_reg": 10.0,
        "random_seed": 42,
        "verbose": False,
        "allow_writing_files": False,
    }

    if params is not None:
        default_params.update(params)


    for fold in temporal_folds_spec:

        train_start = pd.Timestamp(
            fold["train_start"]
        )
        train_end = pd.Timestamp(
            fold["train_end"]
        )

        val_start = pd.Timestamp(
            fold["validation_start"]
        )
        val_end = pd.Timestamp(
            fold["validation_end"]
        )


        train = df[
            (df["date"] >= train_start)
            & (df["date"] <= train_end)
        ].copy()

        val = df[
            (df["date"] >= val_start)
            & (df["date"] <= val_end)
        ].copy()


        X_train = train[features].copy()
        X_val = val[features].copy()

        y_train = train[TARGET].to_numpy(
            dtype=float
        )

        y_val = val[TARGET].to_numpy(
            dtype=float
        )


        # CatBoost expects categorical values to be string-like.
        for col in categorical_features:
            X_train[col] = (
                X_train[col]
                .astype(str)
                .fillna("__MISSING__")
            )

            X_val[col] = (
                X_val[col]
                .astype(str)
                .fillna("__MISSING__")
            )


        model = CatBoostRegressor(
            **default_params
        )


        model.fit(
            X_train,
            y_train,
            cat_features=categorical_features,
        )


        y_pred = model.predict(
            X_val
        )

        y_pred = np.clip(
            y_pred,
            1e-9,
            None
        )


        metrics = {
            "poisson_deviance":
                mean_poisson_deviance(
                    y_val,
                    y_pred
                ),

            "mae":
                mean_absolute_error(
                    y_val,
                    y_pred
                ),

            "rmse":
                np.sqrt(
                    mean_squared_error(
                        y_val,
                        y_pred
                    )
                ),

            "actual_mean_goals":
                y_val.mean(),

            "predicted_mean_goals":
                y_pred.mean(),

            "mean_bias":
                y_pred.mean()
                - y_val.mean(),
        }


        fold_results.append({
            "fold": fold["name"],
            "train_matches": len(train),
            "val_matches": len(val),
            **metrics,
        })


        oof_parts.append(
            pd.DataFrame({
                "match_id":
                    val["match_id"].values,

                "date":
                    val["date"].values,

                "fold":
                    fold["name"],

                "y_true":
                    y_val,

                "y_pred":
                    y_pred,
            })
        )


    fold_results = pd.DataFrame(
        fold_results
    )


    oof = (
        pd.concat(
            oof_parts,
            ignore_index=True
        )
        .sort_values("date")
        .reset_index(drop=True)
    )


    macro_metrics = (
        fold_results[
            [
                "poisson_deviance",
                "mae",
                "rmse",
                "mean_bias",
            ]
        ]
        .mean()
        .to_dict()
    )


    pooled_metrics = {
        "poisson_deviance":
            mean_poisson_deviance(
                oof["y_true"],
                oof["y_pred"]
            ),

        "mae":
            mean_absolute_error(
                oof["y_true"],
                oof["y_pred"]
            ),

        "rmse":
            np.sqrt(
                mean_squared_error(
                    oof["y_true"],
                    oof["y_pred"]
                )
            ),

        "actual_mean_goals":
            oof["y_true"].mean(),

        "predicted_mean_goals":
            oof["y_pred"].mean(),

        "mean_bias":
            oof["y_pred"].mean()
            - oof["y_true"].mean(),
    }


    return {
        "feature_set":
            feature_set_name,

        "params":
            default_params,

        "fold_results":
            fold_results,

        "oof":
            oof,

        "macro_metrics":
            macro_metrics,

        "pooled_metrics":
            pooled_metrics,
    }


catboost_initial_result = (
    evaluate_catboost_poisson(
        feature_set_name="no_season"
    )
)


print("CATBOOST POISSON — INITIAL MODEL")
print("=" * 100)

print(
    catboost_initial_result[
        "fold_results"
    ][
        [
            "fold",
            "actual_mean_goals",
            "predicted_mean_goals",
            "mean_bias",
            "poisson_deviance",
            "mae",
            "rmse",
        ]
    ]
    .round(6)
    .to_string(index=False)
)


print("\nMACRO VALIDATION")
print("=" * 50)

for metric, value in (
    catboost_initial_result[
        "macro_metrics"
    ].items()
):
    print(
        f"{metric:<25} {value:.6f}"
    )


print("\nPOOLED OOF VALIDATION")
print("=" * 50)

for metric, value in (
    catboost_initial_result[
        "pooled_metrics"
    ].items()
):
    print(
        f"{metric:<25} {value:.6f}"
    )


print("\nLOCKED BASELINE")
print("=" * 50)

print(
    "Poisson macro :",
    round(
        POISSON_BASELINE_MACRO,
        6
    )
)

print(
    "Poisson pooled:",
    round(
        POISSON_BASELINE_POOLED,
        6
    )
)


assert len(
    catboost_initial_result["oof"]
) == 340

assert (
    catboost_initial_result[
        "oof"
    ]["match_id"].nunique()
    == 340
)

print(
    "\nCatBoost temporal evaluation: PASSED"
)

CATBOOST POISSON — INITIAL MODEL
  fold  actual_mean_goals  predicted_mean_goals  mean_bias  poisson_deviance      mae     rmse
Fold 1           0.791139              0.866860   0.075721          1.070969 0.688957 0.856848
Fold 2           0.500000              0.799622   0.299622          1.352681 0.784281 0.894276
Fold 3           0.915254              0.430596  -0.484659          1.571094 0.801774 1.051097
Fold 4           0.843137              0.837598  -0.005539          0.971612 0.662833 0.831730

MACRO VALIDATION
poisson_deviance          1.241589
mae                       0.734461
rmse                      0.908488
mean_bias                 -0.028714

POOLED OOF VALIDATION
poisson_deviance          1.202509
mae                       0.724801
rmse                      0.897785
actual_mean_goals         0.758824
predicted_mean_goals      0.772527
mean_bias                 0.013704

LOCKED BASELINE
Poisson macro : 1.113978
Poisson pooled: 1.104881

CatBoost temporal evaluation: PA

In [12]:
# Compare CatBoost feature sets with identical model parameters

catboost_feature_results = {}
comparison_rows = []


for feature_set_name in [
    "context",
    "no_season",
    "full",
]:

    # Reuse the result we already calculated
    if feature_set_name == "no_season":
        result = catboost_initial_result

    else:
        result = evaluate_catboost_poisson(
            feature_set_name=feature_set_name
        )


    catboost_feature_results[
        feature_set_name
    ] = result


    macro = result["macro_metrics"]
    pooled = result["pooled_metrics"]

    fold_dev = (
        result["fold_results"]
        .set_index("fold")[
            "poisson_deviance"
        ]
    )


    comparison_rows.append({
        "feature_set":
            feature_set_name,

        "macro_poisson_deviance":
            macro["poisson_deviance"],

        "pooled_poisson_deviance":
            pooled["poisson_deviance"],

        "pooled_mae":
            pooled["mae"],

        "pooled_rmse":
            pooled["rmse"],

        "pooled_bias":
            pooled["mean_bias"],

        "fold1":
            fold_dev["Fold 1"],

        "fold2":
            fold_dev["Fold 2"],

        "fold3":
            fold_dev["Fold 3"],

        "fold4":
            fold_dev["Fold 4"],
    })


catboost_feature_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values(
        "macro_poisson_deviance"
    )
    .reset_index(drop=True)
)


print(
    "CATBOOST FEATURE-SET COMPARISON"
)

print("=" * 120)

print(
    catboost_feature_comparison
    .round(6)
    .to_string(index=False)
)


print("\nLOCKED BASELINES")
print("=" * 70)

print(
    "Regularized Poisson macro  :",
    round(
        POISSON_BASELINE_MACRO,
        6
    )
)

print(
    "Regularized Poisson pooled :",
    round(
        POISSON_BASELINE_POOLED,
        6
    )
)

print(
    "Historical mean macro      : 1.115870"
)

print(
    "Historical mean pooled     : 1.106694"
)

CATBOOST FEATURE-SET COMPARISON
feature_set  macro_poisson_deviance  pooled_poisson_deviance  pooled_mae  pooled_rmse  pooled_bias    fold1    fold2    fold3    fold4
       full                1.217163                 1.184671    0.722410     0.892376     0.009599 1.062209 1.359651 1.534741 0.912051
    context                1.226503                 1.205117    0.745896     0.910043     0.089644 1.112055 1.380244 1.444747 0.968966
  no_season                1.241589                 1.202509    0.724801     0.897785     0.013704 1.070969 1.352681 1.571094 0.971612

LOCKED BASELINES
Regularized Poisson macro  : 1.113978
Regularized Poisson pooled : 1.104881
Historical mean macro      : 1.115870
Historical mean pooled     : 1.106694


In [14]:
# Diagnose CatBoost tree complexity

CATBOOST_COMPLEXITY_CONFIGS = [
    {
        "name": "shallow_200",
        "iterations": 200,
        "depth": 3,
        "learning_rate": 0.03,
        "l2_leaf_reg": 20.0,
    },
    {
        "name": "shallow_400",
        "iterations": 400,
        "depth": 3,
        "learning_rate": 0.02,
        "l2_leaf_reg": 20.0,
    },
    {
        "name": "depth4_300",
        "iterations": 300,
        "depth": 4,
        "learning_rate": 0.025,
        "l2_leaf_reg": 20.0,
    },
    {
        "name": "depth4_strong_reg",
        "iterations": 400,
        "depth": 4,
        "learning_rate": 0.02,
        "l2_leaf_reg": 40.0,
    },
]

complexity_rows = []
catboost_complexity_details = {}


for config in CATBOOST_COMPLEXITY_CONFIGS:

    name = config["name"]

    params = {
        "iterations": config["iterations"],
        "depth": config["depth"],
        "learning_rate": config["learning_rate"],
        "l2_leaf_reg": config["l2_leaf_reg"],
    }

    result = evaluate_catboost_poisson(
        feature_set_name="full",
        params=params,
    )

    catboost_complexity_details[name] = result

    macro = result["macro_metrics"]
    pooled = result["pooled_metrics"]

    fold_dev = (
        result["fold_results"]
        .set_index("fold")["poisson_deviance"]
    )

    complexity_rows.append({
        "configuration": name,
        "iterations": config["iterations"],
        "depth": config["depth"],
        "learning_rate": config["learning_rate"],
        "l2_leaf_reg": config["l2_leaf_reg"],

        "macro_deviance":
            macro["poisson_deviance"],

        "pooled_deviance":
            pooled["poisson_deviance"],

        "pooled_mae":
            pooled["mae"],

        "pooled_rmse":
            pooled["rmse"],

        "pooled_bias":
            pooled["mean_bias"],

        "fold1":
            fold_dev["Fold 1"],

        "fold2":
            fold_dev["Fold 2"],

        "fold3":
            fold_dev["Fold 3"],

        "fold4":
            fold_dev["Fold 4"],
    })


catboost_complexity_comparison = (
    pd.DataFrame(complexity_rows)
    .sort_values("macro_deviance")
    .reset_index(drop=True)
)


print("CATBOOST COMPLEXITY DIAGNOSTIC — FULL FEATURES")
print("=" * 140)

print(
    catboost_complexity_comparison
    .round(6)
    .to_string(index=False)
)


best_cb_config_name = (
    catboost_complexity_comparison
    .loc[0, "configuration"]
)

best_cb_complexity_result = (
    catboost_complexity_details[
        best_cb_config_name
    ]
)


print("\nBEST CONFIGURATION")
print("=" * 70)

print(
    "Configuration:",
    best_cb_config_name
)

print(
    "Macro deviance:",
    round(
        best_cb_complexity_result[
            "macro_metrics"
        ]["poisson_deviance"],
        6
    )
)

print(
    "Pooled deviance:",
    round(
        best_cb_complexity_result[
            "pooled_metrics"
        ]["poisson_deviance"],
        6
    )
)

print(
    "Pooled bias:",
    round(
        best_cb_complexity_result[
            "pooled_metrics"
        ]["mean_bias"],
        6
    )
)


print("\nBENCHMARK")
print("=" * 70)

print(
    "Regularized Poisson macro :",
    round(POISSON_BASELINE_MACRO, 6)
)

print(
    "Regularized Poisson pooled:",
    round(POISSON_BASELINE_POOLED, 6)
)

CATBOOST COMPLEXITY DIAGNOSTIC — FULL FEATURES
    configuration  iterations  depth  learning_rate  l2_leaf_reg  macro_deviance  pooled_deviance  pooled_mae  pooled_rmse  pooled_bias    fold1    fold2    fold3    fold4
      shallow_200         200      3          0.030         20.0        1.116322         1.101524    0.692115     0.860621    -0.004770 1.040794 1.206382 1.261194 0.956918
      shallow_400         400      3          0.020         20.0        1.123599         1.104121    0.695367     0.862507     0.017492 1.028129 1.243798 1.257176 0.965295
       depth4_300         300      4          0.025         20.0        1.137335         1.120780    0.699298     0.868415     0.053123 1.058385 1.198892 1.328128 0.963936
depth4_strong_reg         400      4          0.020         40.0        1.148804         1.125961    0.703688     0.870710     0.035074 1.041600 1.260053 1.319796 0.973769

BEST CONFIGURATION
Configuration: shallow_200
Macro deviance: 1.116322
Pooled deviance: 1.10

In [16]:
# Step 7E — Refine shallow CatBoost configurations

SHALLOW_CONFIGS = [
    {
        "name": "d2_100",
        "iterations": 100,
        "depth": 2,
        "learning_rate": 0.03,
        "l2_leaf_reg": 20.0,
    },
    {
        "name": "d2_200",
        "iterations": 200,
        "depth": 2,
        "learning_rate": 0.03,
        "l2_leaf_reg": 20.0,
    },
    {
        "name": "d2_300",
        "iterations": 300,
        "depth": 2,
        "learning_rate": 0.02,
        "l2_leaf_reg": 20.0,
    },
    {
        "name": "d3_100",
        "iterations": 100,
        "depth": 3,
        "learning_rate": 0.03,
        "l2_leaf_reg": 20.0,
    },
    {
        "name": "d3_150",
        "iterations": 150,
        "depth": 3,
        "learning_rate": 0.03,
        "l2_leaf_reg": 20.0,
    },
    {
        "name": "d3_200",
        "iterations": 200,
        "depth": 3,
        "learning_rate": 0.03,
        "l2_leaf_reg": 20.0,
    },
    {
        "name": "d3_250",
        "iterations": 250,
        "depth": 3,
        "learning_rate": 0.025,
        "l2_leaf_reg": 20.0,
    },
]

shallow_rows = []
catboost_shallow_details = {}


for config in SHALLOW_CONFIGS:

    params = {
        "iterations": config["iterations"],
        "depth": config["depth"],
        "learning_rate": config["learning_rate"],
        "l2_leaf_reg": config["l2_leaf_reg"],
    }

    result = evaluate_catboost_poisson(
        feature_set_name="full",
        params=params,
    )

    catboost_shallow_details[
        config["name"]
    ] = result

    macro = result["macro_metrics"]
    pooled = result["pooled_metrics"]

    fold_dev = (
        result["fold_results"]
        .set_index("fold")["poisson_deviance"]
    )

    shallow_rows.append({
        "configuration":
            config["name"],

        "iterations":
            config["iterations"],

        "depth":
            config["depth"],

        "learning_rate":
            config["learning_rate"],

        "macro_deviance":
            macro["poisson_deviance"],

        "pooled_deviance":
            pooled["poisson_deviance"],

        "pooled_mae":
            pooled["mae"],

        "pooled_rmse":
            pooled["rmse"],

        "pooled_bias":
            pooled["mean_bias"],

        "fold1":
            fold_dev["Fold 1"],

        "fold2":
            fold_dev["Fold 2"],

        "fold3":
            fold_dev["Fold 3"],

        "fold4":
            fold_dev["Fold 4"],
    })


catboost_shallow_comparison = (
    pd.DataFrame(shallow_rows)
    .sort_values("macro_deviance")
    .reset_index(drop=True)
)


print("CATBOOST SHALLOW REFINEMENT — FULL FEATURES")
print("=" * 130)

print(
    catboost_shallow_comparison
    .round(6)
    .to_string(index=False)
)


BEST_CATBOOST_CONFIG = (
    catboost_shallow_comparison
    .loc[0, "configuration"]
)

BEST_CATBOOST_RESULT = (
    catboost_shallow_details[
        BEST_CATBOOST_CONFIG
    ]
)


print("\nBEST SHALLOW CATBOOST")
print("=" * 70)

print(
    "Configuration:",
    BEST_CATBOOST_CONFIG
)

print(
    "Macro deviance:",
    round(
        BEST_CATBOOST_RESULT[
            "macro_metrics"
        ]["poisson_deviance"],
        6
    )
)

print(
    "Pooled deviance:",
    round(
        BEST_CATBOOST_RESULT[
            "pooled_metrics"
        ]["poisson_deviance"],
        6
    )
)

print(
    "Pooled MAE:",
    round(
        BEST_CATBOOST_RESULT[
            "pooled_metrics"
        ]["mae"],
        6
    )
)

print(
    "Pooled bias:",
    round(
        BEST_CATBOOST_RESULT[
            "pooled_metrics"
        ]["mean_bias"],
        6
    )
)


print("\nPOISSON BENCHMARK")
print("=" * 70)

print(
    "Macro deviance :",
    round(POISSON_BASELINE_MACRO, 6)
)

print(
    "Pooled deviance:",
    round(POISSON_BASELINE_POOLED, 6)
)

CATBOOST SHALLOW REFINEMENT — FULL FEATURES
configuration  iterations  depth  learning_rate  macro_deviance  pooled_deviance  pooled_mae  pooled_rmse  pooled_bias    fold1    fold2    fold3    fold4
       d2_100         100      2          0.030        1.112413         1.102292    0.693109     0.863138     0.032200 1.051434 1.228419 1.193355 0.976445
       d3_200         200      3          0.030        1.116322         1.101524    0.692115     0.860621    -0.004770 1.040794 1.206382 1.261194 0.956918
       d3_100         100      3          0.030        1.124673         1.110185    0.695071     0.865816     0.023672 1.050704 1.218676 1.251500 0.977812
       d2_200         200      2          0.030        1.129687         1.113024    0.698052     0.865878    -0.002122 1.046687 1.226953 1.276285 0.968824
       d3_150         150      3          0.030        1.130660         1.115409    0.695503     0.867334     0.023781 1.053371 1.219431 1.283203 0.966634
       d2_300         300 

In [18]:
# Feature-set comparison for the selected shallow CatBoost

selected_cb_params = {
    "iterations": 100,
    "depth": 2,
    "learning_rate": 0.03,
    "l2_leaf_reg": 20.0,
}


selected_feature_rows = []
selected_feature_details = {}


for feature_set_name in [
    "context",
    "no_season",
    "full",
]:

    result = evaluate_catboost_poisson(
        feature_set_name=feature_set_name,
        params=selected_cb_params,
    )

    selected_feature_details[
        feature_set_name
    ] = result

    macro = result["macro_metrics"]
    pooled = result["pooled_metrics"]

    fold_dev = (
        result["fold_results"]
        .set_index("fold")["poisson_deviance"]
    )


    selected_feature_rows.append({
        "feature_set":
            feature_set_name,

        "macro_deviance":
            macro["poisson_deviance"],

        "pooled_deviance":
            pooled["poisson_deviance"],

        "pooled_mae":
            pooled["mae"],

        "pooled_rmse":
            pooled["rmse"],

        "pooled_bias":
            pooled["mean_bias"],

        "fold1":
            fold_dev["Fold 1"],

        "fold2":
            fold_dev["Fold 2"],

        "fold3":
            fold_dev["Fold 3"],

        "fold4":
            fold_dev["Fold 4"],
    })


selected_cb_feature_comparison = (
    pd.DataFrame(
        selected_feature_rows
    )
    .sort_values("macro_deviance")
    .reset_index(drop=True)
)


print(
    "SELECTED SHALLOW CATBOOST — FEATURE SET COMPARISON"
)

print("=" * 125)

print(
    selected_cb_feature_comparison
    .round(6)
    .to_string(index=False)
)


BEST_CATBOOST_FEATURE_SET = (
    selected_cb_feature_comparison
    .loc[0, "feature_set"]
)

BEST_CATBOOST_RESULT = (
    selected_feature_details[
        BEST_CATBOOST_FEATURE_SET
    ]
)


print("\nCURRENT CATBOOST LEADER")
print("=" * 70)

print(
    "Feature set:",
    BEST_CATBOOST_FEATURE_SET
)

print(
    "Configuration: depth=2, iterations=100, "
    "learning_rate=0.03, l2_leaf_reg=20"
)

print(
    "Macro deviance:",
    round(
        BEST_CATBOOST_RESULT[
            "macro_metrics"
        ]["poisson_deviance"],
        6
    )
)

print(
    "Pooled deviance:",
    round(
        BEST_CATBOOST_RESULT[
            "pooled_metrics"
        ]["poisson_deviance"],
        6
    )
)

print(
    "Pooled MAE:",
    round(
        BEST_CATBOOST_RESULT[
            "pooled_metrics"
        ]["mae"],
        6
    )
)

print(
    "Pooled bias:",
    round(
        BEST_CATBOOST_RESULT[
            "pooled_metrics"
        ]["mean_bias"],
        6
    )
)


print("\nLOCKED POISSON BASELINE")
print("=" * 70)

print(
    "Macro deviance :",
    round(
        POISSON_BASELINE_MACRO,
        6
    )
)

print(
    "Pooled deviance:",
    round(
        POISSON_BASELINE_POOLED,
        6
    )
)

SELECTED SHALLOW CATBOOST — FEATURE SET COMPARISON
feature_set  macro_deviance  pooled_deviance  pooled_mae  pooled_rmse  pooled_bias    fold1    fold2    fold3    fold4
       full        1.112413         1.102292    0.693109     0.863138     0.032200 1.051434 1.228419 1.193355 0.976445
    context        1.124506         1.114419    0.697698     0.869364     0.095229 1.060812 1.268467 1.169804 0.998943
  no_season        1.124595         1.109865    0.696462     0.866236     0.035567 1.043558 1.270573 1.199238 0.985012

CURRENT CATBOOST LEADER
Feature set: full
Configuration: depth=2, iterations=100, learning_rate=0.03, l2_leaf_reg=20
Macro deviance: 1.112413
Pooled deviance: 1.102292
Pooled MAE: 0.693109
Pooled bias: 0.0322

LOCKED POISSON BASELINE
Macro deviance : 1.113978
Pooled deviance: 1.104881


## Check XGBoost and prepare its preprocessing

In [21]:
# Prepare XGBoost Poisson modeling path

import xgboost as xgb

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder


def build_xgboost_preprocessor(
    feature_set
):
    """
    Preprocessing for XGBoost.

    Numeric features pass through unchanged.
    Categorical features are one-hot encoded.

    The transformer must be fitted separately
    inside each temporal training fold.
    """

    numeric_features = [
        feature
        for feature in feature_set
        if feature in NUMERIC_FEATURES
    ]

    categorical_features = [
        feature
        for feature in feature_set
        if feature in CATEGORICAL_FEATURES
    ]


    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                "passthrough",
                numeric_features,
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                ),
                categorical_features,
            ),
        ],
        remainder="drop",
    )


# Start with the full feature representation.
xgb_features = FEATURE_SETS["full"]

xgb_preprocessor_check = (
    build_xgboost_preprocessor(
        xgb_features
    )
)


print("XGBOOST POISSON SETUP")
print("=" * 70)

print(
    "XGBoost version:",
    xgb.__version__
)

print(
    "Feature set:",
    "full"
)

print(
    "Raw features:",
    len(xgb_features)
)

print(
    "Numeric:",
    sum(
        feature in NUMERIC_FEATURES
        for feature in xgb_features
    )
)

print(
    "Categorical:",
    sum(
        feature in CATEGORICAL_FEATURES
        for feature in xgb_features
    )
)

print(
    "Preprocessor fitted:",
    hasattr(
        xgb_preprocessor_check,
        "transformers_"
    )
)


assert not hasattr(
    xgb_preprocessor_check,
    "transformers_"
)

assert len(xgb_features) == 84


print(
    "\nXGBoost preprocessing setup: PASSED"
)

print(
    "No XGBoost model has been fitted."
)

XGBOOST POISSON SETUP
XGBoost version: 3.4.1
Feature set: full
Raw features: 84
Numeric: 77
Categorical: 7
Preprocessor fitted: False

XGBoost preprocessing setup: PASSED
No XGBoost model has been fitted.


In [23]:
# Initial XGBoost Poisson model across temporal folds

from sklearn.pipeline import Pipeline


def evaluate_xgboost_poisson(
    feature_set_name="full",
    params=None,
):
    """
    Evaluate an XGBoost Poisson model using the
    four locked temporal validation folds.
    """

    features = FEATURE_SETS[
        feature_set_name
    ]

    default_params = {
        "objective": "count:poisson",
        "n_estimators": 100,
        "max_depth": 2,
        "learning_rate": 0.03,
        "min_child_weight": 10.0,
        "subsample": 0.9,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.0,
        "reg_lambda": 20.0,
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    if params is not None:
        default_params.update(params)


    fold_results = []
    oof_parts = []


    for fold in temporal_folds_spec:

        train_start = pd.Timestamp(
            fold["train_start"]
        )
        train_end = pd.Timestamp(
            fold["train_end"]
        )

        val_start = pd.Timestamp(
            fold["validation_start"]
        )
        val_end = pd.Timestamp(
            fold["validation_end"]
        )


        train = df[
            (df["date"] >= train_start)
            & (df["date"] <= train_end)
        ].copy()

        val = df[
            (df["date"] >= val_start)
            & (df["date"] <= val_end)
        ].copy()


        X_train = train[features].copy()
        X_val = val[features].copy()

        y_train = train[TARGET].to_numpy(
            dtype=float
        )

        y_val = val[TARGET].to_numpy(
            dtype=float
        )


        # Fresh preprocessing and model for every fold.
        model = Pipeline(
            steps=[
                (
                    "preprocessor",
                    build_xgboost_preprocessor(
                        features
                    )
                ),
                (
                    "model",
                    xgb.XGBRegressor(
                        **default_params
                    )
                ),
            ]
        )


        model.fit(
            X_train,
            y_train,
        )


        y_pred = model.predict(
            X_val
        )

        y_pred = np.clip(
            y_pred,
            1e-9,
            None
        )


        metrics = {
            "poisson_deviance":
                mean_poisson_deviance(
                    y_val,
                    y_pred
                ),

            "mae":
                mean_absolute_error(
                    y_val,
                    y_pred
                ),

            "rmse":
                np.sqrt(
                    mean_squared_error(
                        y_val,
                        y_pred
                    )
                ),

            "actual_mean_goals":
                y_val.mean(),

            "predicted_mean_goals":
                y_pred.mean(),

            "mean_bias":
                y_pred.mean()
                - y_val.mean(),
        }


        fold_results.append({
            "fold": fold["name"],
            "train_matches": len(train),
            "val_matches": len(val),
            **metrics,
        })


        oof_parts.append(
            pd.DataFrame({
                "match_id":
                    val["match_id"].values,

                "date":
                    val["date"].values,

                "fold":
                    fold["name"],

                "y_true":
                    y_val,

                "y_pred":
                    y_pred,
            })
        )


    fold_results = pd.DataFrame(
        fold_results
    )


    oof = (
        pd.concat(
            oof_parts,
            ignore_index=True
        )
        .sort_values("date")
        .reset_index(drop=True)
    )


    macro_metrics = (
        fold_results[
            [
                "poisson_deviance",
                "mae",
                "rmse",
                "mean_bias",
            ]
        ]
        .mean()
        .to_dict()
    )


    pooled_metrics = {
        "poisson_deviance":
            mean_poisson_deviance(
                oof["y_true"],
                oof["y_pred"]
            ),

        "mae":
            mean_absolute_error(
                oof["y_true"],
                oof["y_pred"]
            ),

        "rmse":
            np.sqrt(
                mean_squared_error(
                    oof["y_true"],
                    oof["y_pred"]
                )
            ),

        "actual_mean_goals":
            oof["y_true"].mean(),

        "predicted_mean_goals":
            oof["y_pred"].mean(),

        "mean_bias":
            oof["y_pred"].mean()
            - oof["y_true"].mean(),
    }


    return {
        "feature_set":
            feature_set_name,

        "params":
            default_params,

        "fold_results":
            fold_results,

        "oof":
            oof,

        "macro_metrics":
            macro_metrics,

        "pooled_metrics":
            pooled_metrics,
    }


xgb_initial_result = (
    evaluate_xgboost_poisson(
        feature_set_name="full"
    )
)


print("XGBOOST POISSON — INITIAL MODEL")
print("=" * 100)

print(
    xgb_initial_result[
        "fold_results"
    ][
        [
            "fold",
            "actual_mean_goals",
            "predicted_mean_goals",
            "mean_bias",
            "poisson_deviance",
            "mae",
            "rmse",
        ]
    ]
    .round(6)
    .to_string(index=False)
)


print("\nMACRO VALIDATION")
print("=" * 50)

for metric, value in (
    xgb_initial_result[
        "macro_metrics"
    ].items()
):
    print(
        f"{metric:<25} {value:.6f}"
    )


print("\nPOOLED OOF VALIDATION")
print("=" * 50)

for metric, value in (
    xgb_initial_result[
        "pooled_metrics"
    ].items()
):
    print(
        f"{metric:<25} {value:.6f}"
    )


print("\nCURRENT LEADERS")
print("=" * 60)

print(
    "CatBoost macro : 1.112413"
)

print(
    "CatBoost pooled: 1.102292"
)

print(
    "Poisson macro  :",
    round(
        POISSON_BASELINE_MACRO,
        6
    )
)

print(
    "Poisson pooled :",
    round(
        POISSON_BASELINE_POOLED,
        6
    )
)


assert len(
    xgb_initial_result["oof"]
) == 340

assert (
    xgb_initial_result[
        "oof"
    ]["match_id"].nunique()
    == 340
)


print(
    "\nXGBoost temporal evaluation: PASSED"
)

XGBOOST POISSON — INITIAL MODEL
  fold  actual_mean_goals  predicted_mean_goals  mean_bias  poisson_deviance      mae     rmse
Fold 1           0.791139              0.908197   0.117058          1.055922 0.674213 0.851490
Fold 2           0.500000              0.801086   0.301086          1.285742 0.746809 0.852838
Fold 3           0.915254              0.685963  -0.229292          1.153983 0.752480 0.950064
Fold 4           0.843137              0.872960   0.029823          1.007024 0.659143 0.851262

MACRO VALIDATION
poisson_deviance          1.125668
mae                       0.708161
rmse                      0.876414
mean_bias                 0.054669

POOLED OOF VALIDATION
poisson_deviance          1.114271
mae                       0.700907
rmse                      0.869644
actual_mean_goals         0.758824
predicted_mean_goals      0.841665
mean_bias                 0.082842

CURRENT LEADERS
CatBoost macro : 1.112413
CatBoost pooled: 1.102292
Poisson macro  : 1.113978
Poisson

In [25]:
# Diagnose XGBoost complexity and regularization

XGB_CONFIGS = [
    {
        "name": "d1_100",
        "n_estimators": 100,
        "max_depth": 1,
        "learning_rate": 0.03,
        "min_child_weight": 10.0,
        "reg_lambda": 20.0,
    },
    {
        "name": "d1_200",
        "n_estimators": 200,
        "max_depth": 1,
        "learning_rate": 0.03,
        "min_child_weight": 10.0,
        "reg_lambda": 20.0,
    },
    {
        "name": "d2_50",
        "n_estimators": 50,
        "max_depth": 2,
        "learning_rate": 0.03,
        "min_child_weight": 10.0,
        "reg_lambda": 20.0,
    },
    {
        "name": "d2_100",
        "n_estimators": 100,
        "max_depth": 2,
        "learning_rate": 0.03,
        "min_child_weight": 10.0,
        "reg_lambda": 20.0,
    },
    {
        "name": "d2_150",
        "n_estimators": 150,
        "max_depth": 2,
        "learning_rate": 0.03,
        "min_child_weight": 10.0,
        "reg_lambda": 20.0,
    },
    {
        "name": "d2_100_strong_reg",
        "n_estimators": 100,
        "max_depth": 2,
        "learning_rate": 0.03,
        "min_child_weight": 20.0,
        "reg_lambda": 40.0,
    },
]


xgb_complexity_rows = []
xgb_complexity_details = {}


for config in XGB_CONFIGS:

    params = {
        "n_estimators":
            config["n_estimators"],

        "max_depth":
            config["max_depth"],

        "learning_rate":
            config["learning_rate"],

        "min_child_weight":
            config["min_child_weight"],

        "reg_lambda":
            config["reg_lambda"],
    }

    result = evaluate_xgboost_poisson(
        feature_set_name="full",
        params=params,
    )

    xgb_complexity_details[
        config["name"]
    ] = result


    macro = result["macro_metrics"]
    pooled = result["pooled_metrics"]

    fold_dev = (
        result["fold_results"]
        .set_index("fold")[
            "poisson_deviance"
        ]
    )


    xgb_complexity_rows.append({
        "configuration":
            config["name"],

        "n_estimators":
            config["n_estimators"],

        "max_depth":
            config["max_depth"],

        "min_child_weight":
            config["min_child_weight"],

        "reg_lambda":
            config["reg_lambda"],

        "macro_deviance":
            macro["poisson_deviance"],

        "pooled_deviance":
            pooled["poisson_deviance"],

        "pooled_mae":
            pooled["mae"],

        "pooled_rmse":
            pooled["rmse"],

        "pooled_bias":
            pooled["mean_bias"],

        "fold1":
            fold_dev["Fold 1"],

        "fold2":
            fold_dev["Fold 2"],

        "fold3":
            fold_dev["Fold 3"],

        "fold4":
            fold_dev["Fold 4"],
    })


xgb_complexity_comparison = (
    pd.DataFrame(
        xgb_complexity_rows
    )
    .sort_values("macro_deviance")
    .reset_index(drop=True)
)


print("XGBOOST COMPLEXITY DIAGNOSTIC — FULL FEATURES")
print("=" * 140)

print(
    xgb_complexity_comparison
    .round(6)
    .to_string(index=False)
)


BEST_XGB_CONFIG = (
    xgb_complexity_comparison
    .loc[0, "configuration"]
)

BEST_XGB_RESULT = (
    xgb_complexity_details[
        BEST_XGB_CONFIG
    ]
)


print("\nBEST XGBOOST CONFIGURATION")
print("=" * 70)

print(
    "Configuration:",
    BEST_XGB_CONFIG
)

print(
    "Macro deviance:",
    round(
        BEST_XGB_RESULT[
            "macro_metrics"
        ]["poisson_deviance"],
        6
    )
)

print(
    "Pooled deviance:",
    round(
        BEST_XGB_RESULT[
            "pooled_metrics"
        ]["poisson_deviance"],
        6
    )
)

print(
    "Pooled MAE:",
    round(
        BEST_XGB_RESULT[
            "pooled_metrics"
        ]["mae"],
        6
    )
)

print(
    "Pooled bias:",
    round(
        BEST_XGB_RESULT[
            "pooled_metrics"
        ]["mean_bias"],
        6
    )
)


print("\nCURRENT BENCHMARKS")
print("=" * 70)

print(
    "CatBoost macro : 1.112413"
)

print(
    "CatBoost pooled: 1.102292"
)

print(
    "Poisson macro  :",
    round(
        POISSON_BASELINE_MACRO,
        6
    )
)

print(
    "Poisson pooled :",
    round(
        POISSON_BASELINE_POOLED,
        6
    )
)

XGBOOST COMPLEXITY DIAGNOSTIC — FULL FEATURES
    configuration  n_estimators  max_depth  min_child_weight  reg_lambda  macro_deviance  pooled_deviance  pooled_mae  pooled_rmse  pooled_bias    fold1    fold2    fold3    fold4
            d2_50            50          2              10.0        20.0        1.113103         1.101354    0.701060     0.863700     0.038943 1.043663 1.265543 1.142377 1.000828
           d1_100           100          1              10.0        20.0        1.113714         1.101227    0.696954     0.863479     0.058546 1.040395 1.279607 1.128207 1.006645
           d1_200           200          1              10.0        20.0        1.120362         1.108524    0.697347     0.866933     0.081679 1.050724 1.276079 1.139888 1.014758
d2_100_strong_reg           100          2              20.0        40.0        1.124689         1.112640    0.699352     0.868892     0.081280 1.052624 1.287312 1.149584 1.009237
           d2_100           100          2            

In [27]:
# Refine XGBoost around the low-tree optimum

XGB_LOW_TREE_CONFIGS = [
    {
        "name": "d2_20",
        "n_estimators": 20,
    },
    {
        "name": "d2_30",
        "n_estimators": 30,
    },
    {
        "name": "d2_40",
        "n_estimators": 40,
    },
    {
        "name": "d2_50",
        "n_estimators": 50,
    },
    {
        "name": "d2_60",
        "n_estimators": 60,
    },
    {
        "name": "d2_75",
        "n_estimators": 75,
    },
]


xgb_low_tree_rows = []
xgb_low_tree_details = {}


for config in XGB_LOW_TREE_CONFIGS:

    params = {
        "n_estimators":
            config["n_estimators"],

        "max_depth": 2,

        "learning_rate": 0.03,

        "min_child_weight": 10.0,

        "reg_lambda": 20.0,
    }


    result = evaluate_xgboost_poisson(
        feature_set_name="full",
        params=params,
    )


    xgb_low_tree_details[
        config["name"]
    ] = result


    macro = result["macro_metrics"]
    pooled = result["pooled_metrics"]

    fold_dev = (
        result["fold_results"]
        .set_index("fold")[
            "poisson_deviance"
        ]
    )


    xgb_low_tree_rows.append({
        "configuration":
            config["name"],

        "n_estimators":
            config["n_estimators"],

        "macro_deviance":
            macro["poisson_deviance"],

        "pooled_deviance":
            pooled["poisson_deviance"],

        "pooled_mae":
            pooled["mae"],

        "pooled_rmse":
            pooled["rmse"],

        "pooled_bias":
            pooled["mean_bias"],

        "fold1":
            fold_dev["Fold 1"],

        "fold2":
            fold_dev["Fold 2"],

        "fold3":
            fold_dev["Fold 3"],

        "fold4":
            fold_dev["Fold 4"],
    })


xgb_low_tree_comparison = (
    pd.DataFrame(
        xgb_low_tree_rows
    )
    .sort_values(
        "macro_deviance"
    )
    .reset_index(drop=True)
)


print(
    "XGBOOST LOW-TREE REFINEMENT — FULL FEATURES"
)

print("=" * 125)

print(
    xgb_low_tree_comparison
    .round(6)
    .to_string(index=False)
)


BEST_XGB_LOW_CONFIG = (
    xgb_low_tree_comparison
    .loc[0, "configuration"]
)

BEST_XGB_RESULT = (
    xgb_low_tree_details[
        BEST_XGB_LOW_CONFIG
    ]
)


print("\nBEST XGBOOST")
print("=" * 70)

print(
    "Configuration:",
    BEST_XGB_LOW_CONFIG
)

print(
    "Macro deviance:",
    round(
        BEST_XGB_RESULT[
            "macro_metrics"
        ]["poisson_deviance"],
        6
    )
)

print(
    "Pooled deviance:",
    round(
        BEST_XGB_RESULT[
            "pooled_metrics"
        ]["poisson_deviance"],
        6
    )
)

print(
    "Pooled MAE:",
    round(
        BEST_XGB_RESULT[
            "pooled_metrics"
        ]["mae"],
        6
    )
)

print(
    "Pooled bias:",
    round(
        BEST_XGB_RESULT[
            "pooled_metrics"
        ]["mean_bias"],
        6
    )
)


print("\nCURRENT LEADERS")
print("=" * 70)

print(
    "CatBoost macro : 1.112413"
)

print(
    "CatBoost pooled: 1.102292"
)

print(
    "Poisson macro  :",
    round(
        POISSON_BASELINE_MACRO,
        6
    )
)

print(
    "Poisson pooled :",
    round(
        POISSON_BASELINE_POOLED,
        6
    )
)

XGBOOST LOW-TREE REFINEMENT — FULL FEATURES
configuration  n_estimators  macro_deviance  pooled_deviance  pooled_mae  pooled_rmse  pooled_bias    fold1    fold2    fold3    fold4
        d2_20            20        1.111223         1.100276    0.707199     0.863367    -0.000716 1.046499 1.251935 1.142318 1.004138
        d2_30            30        1.111522         1.099885    0.704994     0.863090     0.013116 1.043743 1.256200 1.143700 1.002445
        d2_40            40        1.111678         1.099835    0.702783     0.863022     0.027140 1.042590 1.260458 1.141838 1.001827
        d2_50            50        1.113103         1.101354    0.701060     0.863700     0.038943 1.043663 1.265543 1.142377 1.000828
        d2_60            60        1.116655         1.104372    0.699902     0.865028     0.049782 1.044594 1.274157 1.145026 1.002843
        d2_75            75        1.119207         1.107318    0.698079     0.866381     0.063242 1.048080 1.278498 1.147131 1.003117

BEST XGBOO

In [29]:
# Bound the low-tree XGBoost optimum

XGB_FINAL_TREE_COUNTS = [
    5,
    10,
    15,
    20,
    25,
    30,
]

xgb_final_tree_rows = []
xgb_final_tree_details = {}


for n_trees in XGB_FINAL_TREE_COUNTS:

    config_name = f"d2_{n_trees}"

    params = {
        "n_estimators": n_trees,
        "max_depth": 2,
        "learning_rate": 0.03,
        "min_child_weight": 10.0,
        "reg_lambda": 20.0,
    }


    # Reuse results we already computed where possible.
    if config_name in xgb_low_tree_details:

        result = xgb_low_tree_details[
            config_name
        ]

    else:

        result = evaluate_xgboost_poisson(
            feature_set_name="full",
            params=params,
        )


    xgb_final_tree_details[
        config_name
    ] = result


    macro = result["macro_metrics"]
    pooled = result["pooled_metrics"]

    fold_dev = (
        result["fold_results"]
        .set_index("fold")[
            "poisson_deviance"
        ]
    )


    xgb_final_tree_rows.append({
        "configuration":
            config_name,

        "n_estimators":
            n_trees,

        "macro_deviance":
            macro["poisson_deviance"],

        "pooled_deviance":
            pooled["poisson_deviance"],

        "pooled_mae":
            pooled["mae"],

        "pooled_rmse":
            pooled["rmse"],

        "pooled_bias":
            pooled["mean_bias"],

        "fold1":
            fold_dev["Fold 1"],

        "fold2":
            fold_dev["Fold 2"],

        "fold3":
            fold_dev["Fold 3"],

        "fold4":
            fold_dev["Fold 4"],
    })


xgb_final_tree_comparison = (
    pd.DataFrame(
        xgb_final_tree_rows
    )
    .sort_values(
        "macro_deviance"
    )
    .reset_index(drop=True)
)


print(
    "XGBOOST FINAL LOW-TREE BOUNDARY CHECK"
)

print("=" * 125)

print(
    xgb_final_tree_comparison
    .round(6)
    .to_string(index=False)
)


BEST_XGB_CONFIG = (
    xgb_final_tree_comparison
    .loc[0, "configuration"]
)

BEST_XGB_RESULT = (
    xgb_final_tree_details[
        BEST_XGB_CONFIG
    ]
)


print("\nCURRENT XGBOOST LEADER")
print("=" * 70)

print(
    "Configuration:",
    BEST_XGB_CONFIG
)

print(
    "Macro deviance:",
    round(
        BEST_XGB_RESULT[
            "macro_metrics"
        ]["poisson_deviance"],
        6
    )
)

print(
    "Pooled deviance:",
    round(
        BEST_XGB_RESULT[
            "pooled_metrics"
        ]["poisson_deviance"],
        6
    )
)

print(
    "Pooled MAE:",
    round(
        BEST_XGB_RESULT[
            "pooled_metrics"
        ]["mae"],
        6
    )
)

print(
    "Pooled bias:",
    round(
        BEST_XGB_RESULT[
            "pooled_metrics"
        ]["mean_bias"],
        6
    )
)


print("\nMODEL LEADERBOARD")
print("=" * 70)

print(
    "XGBoost macro :",
    round(
        BEST_XGB_RESULT[
            "macro_metrics"
        ]["poisson_deviance"],
        6
    )
)

print(
    "CatBoost macro: 1.112413"
)

print(
    "Poisson macro :",
    round(
        POISSON_BASELINE_MACRO,
        6
    )
)

XGBOOST FINAL LOW-TREE BOUNDARY CHECK
configuration  n_estimators  macro_deviance  pooled_deviance  pooled_mae  pooled_rmse  pooled_bias    fold1    fold2    fold3    fold4
        d2_15            15        1.110437         1.099831    0.708044     0.863233    -0.009628 1.047762 1.246705 1.140265 1.007017
        d2_20            20        1.111223         1.100276    0.707199     0.863367    -0.000716 1.046499 1.251935 1.142318 1.004138
        d2_30            30        1.111522         1.099885    0.704994     0.863090     0.013116 1.043743 1.256200 1.143700 1.002445
        d2_25            25        1.111523         1.099992    0.706007     0.863176     0.006454 1.044591 1.253229 1.144694 1.003576
        d2_10            10        1.111669         1.101360    0.709403     0.863934    -0.018475 1.050622 1.244861 1.140390 1.010804
         d2_5             5        1.113223         1.103550    0.711114     0.864899    -0.027666 1.055046 1.242839 1.139995 1.015013

CURRENT XGBOOST 

In [31]:
# Feature-set stability for selected XGBoost

selected_xgb_params = {
    "n_estimators": 15,
    "max_depth": 2,
    "learning_rate": 0.03,
    "min_child_weight": 10.0,
    "reg_lambda": 20.0,
}


xgb_feature_rows = []
xgb_feature_details = {}


for feature_set_name in [
    "context",
    "no_season",
    "full",
]:

    # Reuse the full-feature result already calculated
    if feature_set_name == "full":
        result = BEST_XGB_RESULT

    else:
        result = evaluate_xgboost_poisson(
            feature_set_name=feature_set_name,
            params=selected_xgb_params,
        )


    xgb_feature_details[
        feature_set_name
    ] = result


    macro = result["macro_metrics"]
    pooled = result["pooled_metrics"]

    fold_dev = (
        result["fold_results"]
        .set_index("fold")[
            "poisson_deviance"
        ]
    )


    xgb_feature_rows.append({
        "feature_set":
            feature_set_name,

        "macro_deviance":
            macro["poisson_deviance"],

        "pooled_deviance":
            pooled["poisson_deviance"],

        "pooled_mae":
            pooled["mae"],

        "pooled_rmse":
            pooled["rmse"],

        "pooled_bias":
            pooled["mean_bias"],

        "fold1":
            fold_dev["Fold 1"],

        "fold2":
            fold_dev["Fold 2"],

        "fold3":
            fold_dev["Fold 3"],

        "fold4":
            fold_dev["Fold 4"],
    })


xgb_feature_comparison = (
    pd.DataFrame(
        xgb_feature_rows
    )
    .sort_values(
        "macro_deviance"
    )
    .reset_index(drop=True)
)


print(
    "SELECTED XGBOOST — FEATURE-SET COMPARISON"
)

print("=" * 125)

print(
    xgb_feature_comparison
    .round(6)
    .to_string(index=False)
)


BEST_XGB_FEATURE_SET = (
    xgb_feature_comparison
    .loc[0, "feature_set"]
)

BEST_XGB_RESULT = (
    xgb_feature_details[
        BEST_XGB_FEATURE_SET
    ]
)


print("\nSELECTED XGBOOST")
print("=" * 70)

print(
    "Feature set:",
    BEST_XGB_FEATURE_SET
)

print(
    "Configuration: depth=2, trees=15, "
    "learning_rate=0.03, "
    "min_child_weight=10, "
    "reg_lambda=20"
)

print(
    "Macro deviance:",
    round(
        BEST_XGB_RESULT[
            "macro_metrics"
        ]["poisson_deviance"],
        6
    )
)

print(
    "Pooled deviance:",
    round(
        BEST_XGB_RESULT[
            "pooled_metrics"
        ]["poisson_deviance"],
        6
    )
)

print(
    "Pooled MAE:",
    round(
        BEST_XGB_RESULT[
            "pooled_metrics"
        ]["mae"],
        6
    )
)

print(
    "Pooled bias:",
    round(
        BEST_XGB_RESULT[
            "pooled_metrics"
        ]["mean_bias"],
        6
    )
)


print("\nCURRENT MODEL LEADERS")
print("=" * 70)

print(
    "CatBoost full macro : 1.112413"
)

print(
    "Poisson no-season  :",
    round(
        POISSON_BASELINE_MACRO,
        6
    )
)

SELECTED XGBOOST — FEATURE-SET COMPARISON
feature_set  macro_deviance  pooled_deviance  pooled_mae  pooled_rmse  pooled_bias    fold1    fold2    fold3    fold4
    context        1.109154         1.098756    0.707668     0.862765    -0.008730 1.047050 1.246801 1.136577 1.006188
  no_season        1.110275         1.099830    0.707827     0.863229    -0.007596 1.047761 1.249046 1.138210 1.006081
       full        1.110437         1.099831    0.708044     0.863233    -0.009628 1.047762 1.246705 1.140265 1.007017

SELECTED XGBOOST
Feature set: context
Configuration: depth=2, trees=15, learning_rate=0.03, min_child_weight=10, reg_lambda=20
Macro deviance: 1.109154
Pooled deviance: 1.098756
Pooled MAE: 0.707668
Pooled bias: -0.00873

CURRENT MODEL LEADERS
CatBoost full macro : 1.112413
Poisson no-season  : 1.113978


## Test validation-only model blending

In [34]:
# Validation-only blend of XGBoost and CatBoost

# Best XGBoost:
# context features, depth=2, 15 trees
xgb_oof = (
    BEST_XGB_RESULT["oof"]
    .rename(
        columns={
            "y_pred": "xgb_pred"
        }
    )
)


# Best CatBoost:
# full features, depth=2, 100 trees
catboost_oof = (
    selected_feature_details["full"]["oof"]
    .rename(
        columns={
            "y_pred": "catboost_pred"
        }
    )
)


blend_oof = (
    xgb_oof[
        [
            "match_id",
            "date",
            "fold",
            "y_true",
            "xgb_pred",
        ]
    ]
    .merge(
        catboost_oof[
            [
                "match_id",
                "catboost_pred",
            ]
        ],
        on="match_id",
        how="inner",
        validate="one_to_one",
    )
)


assert len(blend_oof) == 340
assert blend_oof["match_id"].nunique() == 340


blend_rows = []


# Weight = contribution from XGBoost.
# Example:
# 0.75 = 75% XGBoost + 25% CatBoost.
for xgb_weight in np.arange(
    0.0,
    1.01,
    0.10
):

    catboost_weight = (
        1.0 - xgb_weight
    )

    y_pred = (
        xgb_weight
        * blend_oof["xgb_pred"]
        +
        catboost_weight
        * blend_oof["catboost_pred"]
    )

    y_pred = np.clip(
        y_pred,
        1e-9,
        None,
    )


    fold_deviances = []

    for fold_name, fold_data in (
        blend_oof.groupby(
            "fold",
            sort=False,
        )
    ):

        fold_pred = (
            xgb_weight
            * fold_data["xgb_pred"]
            +
            catboost_weight
            * fold_data["catboost_pred"]
        )

        fold_pred = np.clip(
            fold_pred,
            1e-9,
            None,
        )

        fold_deviances.append(
            mean_poisson_deviance(
                fold_data["y_true"],
                fold_pred,
            )
        )


    blend_rows.append({
        "xgb_weight":
            xgb_weight,

        "catboost_weight":
            catboost_weight,

        "macro_deviance":
            np.mean(
                fold_deviances
            ),

        "pooled_deviance":
            mean_poisson_deviance(
                blend_oof["y_true"],
                y_pred,
            ),

        "pooled_mae":
            mean_absolute_error(
                blend_oof["y_true"],
                y_pred,
            ),

        "pooled_rmse":
            np.sqrt(
                mean_squared_error(
                    blend_oof["y_true"],
                    y_pred,
                )
            ),

        "pooled_bias":
            (
                y_pred.mean()
                - blend_oof[
                    "y_true"
                ].mean()
            ),
    })


blend_comparison = (
    pd.DataFrame(blend_rows)
    .sort_values(
        "macro_deviance"
    )
    .reset_index(drop=True)
)


print(
    "XGBOOST + CATBOOST OOF BLEND"
)

print("=" * 100)

print(
    blend_comparison
    .round(6)
    .to_string(index=False)
)


best_blend = (
    blend_comparison.iloc[0]
)


print("\nBEST VALIDATION BLEND")
print("=" * 70)

print(
    "XGBoost weight :",
    round(
        best_blend[
            "xgb_weight"
        ],
        2
    )
)

print(
    "CatBoost weight:",
    round(
        best_blend[
            "catboost_weight"
        ],
        2
    )
)

print(
    "Macro deviance :",
    round(
        best_blend[
            "macro_deviance"
        ],
        6
    )
)

print(
    "Pooled deviance:",
    round(
        best_blend[
            "pooled_deviance"
        ],
        6
    )
)

print(
    "Pooled MAE     :",
    round(
        best_blend[
            "pooled_mae"
        ],
        6
    )
)

print(
    "Pooled bias    :",
    round(
        best_blend[
            "pooled_bias"
        ],
        6
    )
)


print("\nSINGLE-MODEL BENCHMARKS")
print("=" * 70)

print(
    "XGBoost context macro: 1.109154"
)

print(
    "CatBoost full macro  : 1.112413"
)

print(
    "Poisson macro        :",
    round(
        POISSON_BASELINE_MACRO,
        6
    )
)

XGBOOST + CATBOOST OOF BLEND
 xgb_weight  catboost_weight  macro_deviance  pooled_deviance  pooled_mae  pooled_rmse  pooled_bias
        0.6              0.4        1.106496         1.096307    0.701696     0.861328     0.007642
        0.7              0.3        1.106659         1.096422    0.703189     0.861489     0.003549
        0.5              0.5        1.106663         1.096515    0.700203     0.861300     0.011735
        0.8              0.2        1.107154         1.096864    0.704682     0.861783    -0.000544
        0.4              0.6        1.107156         1.097041    0.698710     0.861403     0.015828
        0.3              0.7        1.107977         1.097883    0.697217     0.861639     0.019921
        0.9              0.1        1.107984         1.097640    0.706175     0.862208    -0.004637
        0.2              0.8        1.109125         1.099040    0.695724     0.862007     0.024014
        1.0              0.0        1.109154         1.098756    0.7076

In [36]:
# Inspect fold stability of the selected 60/40 ensemble

SELECTED_XGB_WEIGHT = 0.60
SELECTED_CATBOOST_WEIGHT = 0.40


blend_oof["blend_pred"] = (
    SELECTED_XGB_WEIGHT
    * blend_oof["xgb_pred"]
    +
    SELECTED_CATBOOST_WEIGHT
    * blend_oof["catboost_pred"]
)


fold_comparison_rows = []


for fold_name, fold_data in (
    blend_oof.groupby(
        "fold",
        sort=False,
    )
):

    y_true = fold_data[
        "y_true"
    ].to_numpy()

    xgb_pred = fold_data[
        "xgb_pred"
    ].to_numpy()

    cat_pred = fold_data[
        "catboost_pred"
    ].to_numpy()

    blend_pred = fold_data[
        "blend_pred"
    ].to_numpy()


    fold_comparison_rows.append({
        "fold":
            fold_name,

        "matches":
            len(fold_data),

        "actual_mean":
            y_true.mean(),

        "xgb_mean":
            xgb_pred.mean(),

        "catboost_mean":
            cat_pred.mean(),

        "blend_mean":
            blend_pred.mean(),

        "xgb_deviance":
            mean_poisson_deviance(
                y_true,
                xgb_pred,
            ),

        "catboost_deviance":
            mean_poisson_deviance(
                y_true,
                cat_pred,
            ),

        "blend_deviance":
            mean_poisson_deviance(
                y_true,
                blend_pred,
            ),

        "xgb_mae":
            mean_absolute_error(
                y_true,
                xgb_pred,
            ),

        "catboost_mae":
            mean_absolute_error(
                y_true,
                cat_pred,
            ),

        "blend_mae":
            mean_absolute_error(
                y_true,
                blend_pred,
            ),

        "blend_bias":
            blend_pred.mean()
            - y_true.mean(),
    })


ensemble_fold_comparison = (
    pd.DataFrame(
        fold_comparison_rows
    )
)


print(
    "SELECTED ENSEMBLE — FOLD STABILITY"
)

print("=" * 150)

print(
    ensemble_fold_comparison
    .round(6)
    .to_string(index=False)
)


# Count fold-by-fold wins.
ensemble_fold_comparison[
    "blend_vs_xgb"
] = (
    ensemble_fold_comparison[
        "blend_deviance"
    ]
    -
    ensemble_fold_comparison[
        "xgb_deviance"
    ]
)

ensemble_fold_comparison[
    "blend_vs_catboost"
] = (
    ensemble_fold_comparison[
        "blend_deviance"
    ]
    -
    ensemble_fold_comparison[
        "catboost_deviance"
    ]
)


print("\nENSEMBLE IMPROVEMENT BY FOLD")
print("=" * 80)

print(
    ensemble_fold_comparison[
        [
            "fold",
            "blend_vs_xgb",
            "blend_vs_catboost",
        ]
    ]
    .round(6)
    .to_string(index=False)
)


print("\nSUMMARY")
print("=" * 80)

print(
    "Folds beating XGBoost:",
    int(
        (
            ensemble_fold_comparison[
                "blend_vs_xgb"
            ] < 0
        ).sum()
    ),
    "/ 4",
)

print(
    "Folds beating CatBoost:",
    int(
        (
            ensemble_fold_comparison[
                "blend_vs_catboost"
            ] < 0
        ).sum()
    ),
    "/ 4",
)

print(
    "Worst ensemble fold deviance:",
    round(
        ensemble_fold_comparison[
            "blend_deviance"
        ].max(),
        6,
    )
)

print(
    "Macro ensemble deviance:",
    round(
        ensemble_fold_comparison[
            "blend_deviance"
        ].mean(),
        6,
    )
)

SELECTED ENSEMBLE — FOLD STABILITY
  fold  matches  actual_mean  xgb_mean  catboost_mean  blend_mean  xgb_deviance  catboost_deviance  blend_deviance  xgb_mae  catboost_mae  blend_mae  blend_bias
Fold 1      158     0.791139  0.761093       0.842766    0.793762      1.047050           1.051434        1.044600 0.691099      0.671302   0.682861    0.002623
Fold 2       72     0.500000  0.741439       0.734222    0.738552      1.246801           1.228419        1.238840 0.727289      0.722731   0.725466    0.238552
Fold 3       59     0.915254  0.714660       0.654597    0.690635      1.136577           1.193355        1.153896 0.755140      0.758878   0.756635   -0.224619
Fold 4       51     0.843137  0.769229       0.868740    0.809033      1.006188           0.976445        0.988650 0.676377      0.642764   0.662932   -0.034104

ENSEMBLE IMPROVEMENT BY FOLD
  fold  blend_vs_xgb  blend_vs_catboost
Fold 1     -0.002450          -0.006834
Fold 2     -0.007961           0.010421
Fold 3    

## Save the model contract and OOF Predictions

In [39]:
# Persist Phase 7 main-model selection artifacts

ARTIFACTS_DIR = Path("../artifacts")
ARTIFACTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------
# Final selected model specifications
# ---------------------------------------------------------

main_model_spec = {
    "phase": "Phase 7 - Main Models",

    "selection_policy": {
        "primary_metric":
            "macro_poisson_deviance",

        "secondary_checks": [
            "pooled_poisson_deviance",
            "mae",
            "rmse",
            "mean_bias",
            "fold_stability",
        ],

        "final_test_used_for_selection":
            False,
    },


    "selected_ensemble": {
        "xgboost_weight":
            0.60,

        "catboost_weight":
            0.40,

        "macro_poisson_deviance":
            1.106496,

        "pooled_poisson_deviance":
            1.096307,

        "pooled_mae":
            0.701696,

        "pooled_bias":
            0.007642,
    },


    "xgboost": {
        "feature_set":
            "context",

        "excluded_identity_categoricals": [
            "season",
            "team",
            "opponent",
            "competition",
        ],

        "params": {
            "objective":
                "count:poisson",

            "n_estimators":
                15,

            "max_depth":
                2,

            "learning_rate":
                0.03,

            "min_child_weight":
                10.0,

            "subsample":
                0.9,

            "colsample_bytree":
                0.8,

            "reg_alpha":
                0.0,

            "reg_lambda":
                20.0,

            "random_state":
                42,

            "tree_method":
                "hist",
        },

        "macro_poisson_deviance":
            1.109154,

        "pooled_poisson_deviance":
            1.098756,

        "pooled_mae":
            0.707668,

        "pooled_bias":
            -0.008730,
    },


    "catboost": {
        "feature_set":
            "full",

        "params": {
            "loss_function":
                "Poisson",

            "iterations":
                100,

            "depth":
                2,

            "learning_rate":
                0.03,

            "l2_leaf_reg":
                20.0,

            "random_seed":
                42,

            "verbose":
                False,

            "allow_writing_files":
                False,
        },

        "macro_poisson_deviance":
            1.112413,

        "pooled_poisson_deviance":
            1.102292,

        "pooled_mae":
            0.693109,

        "pooled_bias":
            0.032200,
    },


    "statistical_baseline": {
        "model":
            "PoissonRegressor",

        "feature_set":
            "no_season",

        "alpha":
            50.0,

        "macro_poisson_deviance":
            POISSON_BASELINE_MACRO,

        "pooled_poisson_deviance":
            POISSON_BASELINE_POOLED,
    },


    "validation_notes": [
        (
            "The selected ensemble beat the best "
            "single XGBoost model in 3 of 4 folds."
        ),
        (
            "Fold 2 remained a difficult low-scoring "
            "regime and was overpredicted."
        ),
        (
            "Fold 3 remained a difficult rebound "
            "regime and was underpredicted."
        ),
        (
            "Further tuning was stopped to reduce "
            "validation-set overfitting risk."
        ),
        (
            "The final 2025-2026 test set remained "
            "untouched during model selection."
        ),
    ],
}


# ---------------------------------------------------------
# Save selected ensemble OOF predictions
# ---------------------------------------------------------

ensemble_oof = (
    blend_oof[
        [
            "match_id",
            "date",
            "fold",
            "y_true",
            "xgb_pred",
            "catboost_pred",
            "blend_pred",
        ]
    ]
    .sort_values("date")
    .reset_index(drop=True)
)


ensemble_oof_path = (
    ARTIFACTS_DIR
    / "main_models_oof.csv"
)

ensemble_oof.to_csv(
    ensemble_oof_path,
    index=False,
)


# ---------------------------------------------------------
# Save model specification
# ---------------------------------------------------------

main_model_spec_path = (
    ARTIFACTS_DIR
    / "main_model_spec.json"
)

with open(
    main_model_spec_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        main_model_spec,
        f,
        indent=2,
    )


# ---------------------------------------------------------
# Integrity checks
# ---------------------------------------------------------

assert len(ensemble_oof) == 340

assert (
    ensemble_oof[
        "match_id"
    ].nunique()
    == 340
)

assert (
    ensemble_oof[
        "date"
    ].max()
    < FINAL_TEST_START
)

assert np.isfinite(
    ensemble_oof[
        [
            "xgb_pred",
            "catboost_pred",
            "blend_pred",
        ]
    ].to_numpy()
).all()


print(
    "PHASE 7 ARTIFACTS SAVED"
)

print("=" * 70)

print(
    "Model specification:",
    main_model_spec_path
)

print(
    "OOF predictions    :",
    ensemble_oof_path
)

print(
    "\nOOF rows           :",
    len(ensemble_oof)
)

print(
    "Unique match IDs   :",
    ensemble_oof[
        "match_id"
    ].nunique()
)

print(
    "OOF period         :",
    ensemble_oof["date"].min().date(),
    "→",
    ensemble_oof["date"].max().date(),
)

print(
    "\nSelected ensemble  :",
    "60% XGBoost + 40% CatBoost"
)

print(
    "Macro deviance     :",
    1.106496
)

print(
    "Pooled deviance    :",
    1.096307
)

print(
    "\nFinal test used     :",
    False
)

print(
    "\nPhase 7 persistence: PASSED"
)

PHASE 7 ARTIFACTS SAVED
Model specification: ..\artifacts\main_model_spec.json
OOF predictions    : ..\artifacts\main_models_oof.csv

OOF rows           : 340
Unique match IDs   : 340
OOF period         : 2018-08-18 → 2024-12-06

Selected ensemble  : 60% XGBoost + 40% CatBoost
Macro deviance     : 1.106496
Pooled deviance    : 1.096307

Final test used     : False

Phase 7 persistence: PASSED
